# 08 — Evaluation & Insights

**Objective**: Deep evaluation of the best LightGBM model with SHAP explanations, error analysis, business metrics, and calibration assessment.

## What we cover

| Section | Content |
|---------|--------|
| **Performance Overview** | Classification report, confusion matrix, ROC, PR curves |
| **Threshold Analysis** | Optimal thresholds for F1, target recall, target precision |
| **SHAP Analysis** | Beeswarm, bar, dependence, force, and waterfall plots |
| **Error Analysis** | False positive/negative patterns, feature distributions |
| **Business Metrics** | Cost model, expected loss curve, profit-optimal threshold |
| **Calibration** | Calibration curve, Brier score, CalibratedClassifierCV |

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from pathlib import Path
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

import mlflow

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Project paths
PROJECT_ROOT = Path("..").resolve()
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# MLFlow setup
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT}/mlruns/mlflow.db")

print(f"Project root:  {PROJECT_ROOT}")
print(f"Features dir:  {FEATURES_DIR}")
print(f"Models dir:    {MODELS_DIR}")
print(f"Figures dir:   {FIGURES_DIR}")

In [ ]:
# Load gold layer features
df = pd.read_parquet(FEATURES_DIR / "train_features.parquet")

print(f"Gold layer shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Default rate: {df['TARGET'].mean()*100:.2f}%")

# Separate target and features
y = df["TARGET"]
X = df.drop(columns=["SK_ID_CURR", "TARGET"])

# Identify column types
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"\nFeatures: {len(numeric_cols)} numeric, {len(categorical_cols)} categorical")

In [ ]:
# Same train/test split as NB07 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train set: {X_train.shape[0]:>8,} rows  (default rate: {y_train.mean()*100:.2f}%)")
print(f"Test set:  {X_test.shape[0]:>8,} rows  (default rate: {y_test.mean()*100:.2f}%)")

In [ ]:
# Encode categoricals same way as NB07 (OrdinalEncoder)
if categorical_cols:
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
    X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])
    print(f"Encoded {len(categorical_cols)} categorical features with OrdinalEncoder")
else:
    print("No categorical features to encode")

feature_names = X_train.columns.tolist()
print(f"Total features: {len(feature_names)}")

In [ ]:
# Load best model from NB07
model_path = MODELS_DIR / "best_lgbm.txt"
booster = lgb.Booster(model_file=str(model_path))

print(f"Loaded model from: {model_path}")
print(f"Number of trees: {booster.num_trees()}")
print(f"Number of features: {booster.num_feature()}")

## 2. Model Performance Overview

In [ ]:
# Compute predictions on test set
y_proba = booster.predict(X_test)
y_pred = (y_proba >= 0.5).astype(int)

# Metrics
auc_roc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("Model Performance on Test Set")
print("=" * 55)
print(f"  AUC-ROC:   {auc_roc:.4f}")
print(f"  PR-AUC:    {pr_auc:.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"\nClassification Report (threshold = 0.5):")
print(classification_report(y_test, y_pred, target_names=["No Default", "Default"]))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt=",d", cmap="Blues",
    xticklabels=["No Default", "Default"],
    yticklabels=["No Default", "Default"],
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (threshold = 0.5)")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_confusion_matrix_default.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_confusion_matrix_default.png'}")
plt.show()

In [ ]:
# ROC and Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, label=f"LightGBM (AUC = {auc_roc:.4f})", color="steelblue", lw=2)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")

# Precision-Recall Curve
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_test, y_proba)
axes[1].plot(recall_arr, precision_arr, label=f"LightGBM (PR-AUC = {pr_auc:.4f})", color="darkorange", lw=2)
axes[1].axhline(y=y_test.mean(), color="k", linestyle="--", alpha=0.5, label=f"Baseline ({y_test.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(loc="upper right")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_roc_pr_curves.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_roc_pr_curves.png'}")
plt.show()

## 3. Threshold Analysis

In [ ]:
# Compute precision, recall, F1 across thresholds
thresholds = np.arange(0.01, 1.0, 0.01)

precisions = []
recalls = []
f1_scores = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    if preds.sum() == 0:
        precisions.append(0.0)
        recalls.append(0.0)
        f1_scores.append(0.0)
    else:
        precisions.append(precision_score(y_test, preds, zero_division=0))
        recalls.append(recall_score(y_test, preds, zero_division=0))
        f1_scores.append(f1_score(y_test, preds, zero_division=0))

precisions = np.array(precisions)
recalls = np.array(recalls)
f1_scores = np.array(f1_scores)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, precisions, label="Precision", color="steelblue", lw=2)
ax.plot(thresholds, recalls, label="Recall", color="darkorange", lw=2)
ax.plot(thresholds, f1_scores, label="F1 Score", color="green", lw=2)
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision, Recall, F1 vs. Classification Threshold")
ax.legend(loc="center left")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_threshold_analysis.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_threshold_analysis.png'}")
plt.show()

In [ ]:
# Find optimal thresholds

# 1. Maximum F1
best_f1_idx = np.argmax(f1_scores)
thresh_best_f1 = thresholds[best_f1_idx]

# 2. Target recall of 0.7 (catch 70% of defaults)
recall_target = 0.7
valid_recall_mask = recalls >= recall_target
if valid_recall_mask.any():
    # Among thresholds that achieve >= 70% recall, pick the highest threshold (highest precision)
    thresh_recall_70 = thresholds[valid_recall_mask].max()
else:
    thresh_recall_70 = thresholds[np.argmin(np.abs(recalls - recall_target))]

# 3. Target precision of 0.3
precision_target = 0.3
valid_precision_mask = precisions >= precision_target
if valid_precision_mask.any():
    # Among thresholds that achieve >= 30% precision, pick the lowest threshold (highest recall)
    thresh_precision_30 = thresholds[valid_precision_mask].min()
else:
    thresh_precision_30 = thresholds[np.argmin(np.abs(precisions - precision_target))]

print("Optimal Thresholds")
print("=" * 65)

for label, t in [("Max F1", thresh_best_f1), ("Target Recall >= 0.7", thresh_recall_70), ("Target Precision >= 0.3", thresh_precision_30)]:
    preds_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, preds_t, zero_division=0)
    r = recall_score(y_test, preds_t, zero_division=0)
    f = f1_score(y_test, preds_t, zero_division=0)
    print(f"\n  {label}")
    print(f"    Threshold:  {t:.2f}")
    print(f"    Precision:  {p:.4f}")
    print(f"    Recall:     {r:.4f}")
    print(f"    F1 Score:   {f:.4f}")

In [ ]:
# Confusion matrices at different thresholds side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

threshold_labels = [
    (f"Max F1 (t={thresh_best_f1:.2f})", thresh_best_f1),
    (f"Recall >= 0.7 (t={thresh_recall_70:.2f})", thresh_recall_70),
    (f"Precision >= 0.3 (t={thresh_precision_30:.2f})", thresh_precision_30),
]

for ax, (label, t) in zip(axes, threshold_labels):
    preds_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, preds_t)
    sns.heatmap(
        cm_t, annot=True, fmt=",d", cmap="Blues",
        xticklabels=["No Default", "Default"],
        yticklabels=["No Default", "Default"],
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(label)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_confusion_matrices_thresholds.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_confusion_matrices_thresholds.png'}")
plt.show()

## 4. SHAP Analysis (Deep Dive)

In [ ]:
# TreeExplainer on test set (sample if too large)
SHAP_SAMPLE_SIZE = 5000

if X_test.shape[0] > SHAP_SAMPLE_SIZE:
    np.random.seed(RANDOM_STATE)
    shap_idx = np.random.choice(X_test.index, size=SHAP_SAMPLE_SIZE, replace=False)
    X_shap = X_test.loc[shap_idx]
    y_shap = y_test.loc[shap_idx]
    print(f"Using random sample of {SHAP_SAMPLE_SIZE} observations for SHAP")
else:
    X_shap = X_test
    y_shap = y_test
    print(f"Using full test set ({X_test.shape[0]} observations) for SHAP")

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (log-odds): {explainer.expected_value:.4f}")

In [ ]:
# Summary plot (beeswarm) — top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, max_display=20, show=False)
plt.title("SHAP Summary Plot (Beeswarm) — Top 20 Features")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_shap_beeswarm.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_shap_beeswarm.png'}")
plt.show()

In [ ]:
# Bar plot — mean |SHAP| for top 20
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type="bar", max_display=20, show=False)
plt.title("Mean |SHAP Value| — Top 20 Features")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_shap_bar.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_shap_bar.png'}")
plt.show()

In [ ]:
# Dependence plots — top 4 features (auto interaction)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_features_idx = np.argsort(-mean_abs_shap)[:4]
top_features = [feature_names[i] for i in top_features_idx]

print(f"Top 4 features by mean |SHAP|: {top_features}\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (feat, ax) in enumerate(zip(top_features, axes.ravel())):
    shap.dependence_plot(
        feat, shap_values, X_shap,
        interaction_index="auto",
        ax=ax, show=False,
    )
    ax.set_title(f"SHAP Dependence — {feat}")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_shap_dependence.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_shap_dependence.png'}")
plt.show()

In [ ]:
# Force and waterfall plots for individual predictions
# Find a correctly classified default and a correctly classified non-default
y_pred_shap = (booster.predict(X_shap) >= thresh_best_f1).astype(int)

# True positive (correctly classified default)
tp_mask = (y_shap == 1) & (y_pred_shap == 1)
if tp_mask.sum() > 0:
    tp_idx = tp_mask[tp_mask].index[0]
    tp_pos = list(X_shap.index).index(tp_idx)
    print(f"True Positive example: index={tp_idx}, position={tp_pos}")
    print(f"  Actual: {y_shap.loc[tp_idx]}, Predicted prob: {booster.predict(X_shap)[tp_pos]:.4f}")
else:
    print("No true positives found in SHAP sample")
    tp_pos = None

# True negative (correctly classified non-default)
tn_mask = (y_shap == 0) & (y_pred_shap == 0)
if tn_mask.sum() > 0:
    tn_idx = tn_mask[tn_mask].index[0]
    tn_pos = list(X_shap.index).index(tn_idx)
    print(f"\nTrue Negative example: index={tn_idx}, position={tn_pos}")
    print(f"  Actual: {y_shap.loc[tn_idx]}, Predicted prob: {booster.predict(X_shap)[tn_pos]:.4f}")
else:
    print("No true negatives found in SHAP sample")
    tn_pos = None

In [ ]:
# Force plots
shap.initjs()

if tp_pos is not None:
    print("Force Plot — True Positive (correctly identified default):")
    display(shap.force_plot(
        explainer.expected_value, shap_values[tp_pos, :], X_shap.iloc[tp_pos],
        matplotlib=True, show=False,
    ))
    plt.savefig(FIGURES_DIR / "08_shap_force_tp.png", dpi=150, bbox_inches="tight")
    print(f"Figure saved: {FIGURES_DIR / '08_shap_force_tp.png'}")
    plt.show()

if tn_pos is not None:
    print("\nForce Plot — True Negative (correctly identified non-default):")
    display(shap.force_plot(
        explainer.expected_value, shap_values[tn_pos, :], X_shap.iloc[tn_pos],
        matplotlib=True, show=False,
    ))
    plt.savefig(FIGURES_DIR / "08_shap_force_tn.png", dpi=150, bbox_inches="tight")
    print(f"Figure saved: {FIGURES_DIR / '08_shap_force_tn.png'}")
    plt.show()

In [ ]:
# Waterfall plots
shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=np.full(shap_values.shape[0], explainer.expected_value),
    data=X_shap.values,
    feature_names=feature_names,
)

if tp_pos is not None:
    print("Waterfall Plot — True Positive (correctly identified default):")
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.plots.waterfall(shap_explanation[tp_pos], max_display=15, show=False)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "08_shap_waterfall_tp.png", dpi=150, bbox_inches="tight")
    print(f"Figure saved: {FIGURES_DIR / '08_shap_waterfall_tp.png'}")
    plt.show()

if tn_pos is not None:
    print("\nWaterfall Plot — True Negative (correctly identified non-default):")
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.plots.waterfall(shap_explanation[tn_pos], max_display=15, show=False)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "08_shap_waterfall_tn.png", dpi=150, bbox_inches="tight")
    print(f"Figure saved: {FIGURES_DIR / '08_shap_waterfall_tn.png'}")
    plt.show()

## 5. Error Analysis

In [ ]:
# Identify FP and FN at optimal F1 threshold
y_pred_opt = (y_proba >= thresh_best_f1).astype(int)

tp_mask_full = (y_test == 1) & (y_pred_opt == 1)
tn_mask_full = (y_test == 0) & (y_pred_opt == 0)
fp_mask_full = (y_test == 0) & (y_pred_opt == 1)
fn_mask_full = (y_test == 1) & (y_pred_opt == 0)

print(f"Error Analysis at Optimal F1 Threshold ({thresh_best_f1:.2f})")
print("=" * 55)
print(f"  True Positives:   {tp_mask_full.sum():>7,}  (defaults caught)")
print(f"  True Negatives:   {tn_mask_full.sum():>7,}  (non-defaults correctly passed)")
print(f"  False Positives:  {fp_mask_full.sum():>7,}  (non-defaults incorrectly flagged)")
print(f"  False Negatives:  {fn_mask_full.sum():>7,}  (defaults missed)")

In [ ]:
# Compare feature distributions: TP vs FN (what defaults did we miss?)
# Use top 5 features by SHAP importance
top5_features = [feature_names[i] for i in np.argsort(-mean_abs_shap)[:5]]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle("Defaults: True Positives vs False Negatives (top 5 features)", fontsize=14, y=1.02)

for ax, feat in zip(axes, top5_features):
    data_tp = X_test.loc[tp_mask_full, feat].dropna()
    data_fn = X_test.loc[fn_mask_full, feat].dropna()

    plot_data = pd.DataFrame({
        feat: pd.concat([data_tp, data_fn]),
        "Group": ["TP (caught)"] * len(data_tp) + ["FN (missed)"] * len(data_fn),
    })

    sns.boxplot(data=plot_data, x="Group", y=feat, ax=ax, palette=["#2ecc71", "#e74c3c"])
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_error_tp_vs_fn.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_error_tp_vs_fn.png'}")
plt.show()

In [ ]:
# Compare feature distributions: TN vs FP (what non-defaults did we flag?)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle("Non-Defaults: True Negatives vs False Positives (top 5 features)", fontsize=14, y=1.02)

for ax, feat in zip(axes, top5_features):
    data_tn = X_test.loc[tn_mask_full, feat].dropna()
    data_fp = X_test.loc[fp_mask_full, feat].dropna()

    # Sample TN if too many for plotting
    if len(data_tn) > 5000:
        data_tn = data_tn.sample(5000, random_state=RANDOM_STATE)

    plot_data = pd.DataFrame({
        feat: pd.concat([data_tn, data_fp]),
        "Group": ["TN (correct)"] * len(data_tn) + ["FP (flagged)"] * len(data_fp),
    })

    sns.boxplot(data=plot_data, x="Group", y=feat, ax=ax, palette=["#3498db", "#e67e22"])
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_error_tn_vs_fp.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_error_tn_vs_fp.png'}")
plt.show()

In [ ]:
# Common patterns in misclassifications
print("False Negatives (missed defaults) — median feature values vs True Positives")
print("=" * 70)

comparison_fn = pd.DataFrame({
    "TP (caught)": X_test.loc[tp_mask_full, top5_features].median(),
    "FN (missed)": X_test.loc[fn_mask_full, top5_features].median(),
})
comparison_fn["Difference"] = comparison_fn["FN (missed)"] - comparison_fn["TP (caught)"]
comparison_fn["%Diff"] = (comparison_fn["Difference"] / comparison_fn["TP (caught)"].abs().clip(lower=1e-6) * 100).round(1)
print(comparison_fn.to_string(float_format="{:.2f}".format))

print("\n\nFalse Positives (incorrectly flagged) — median feature values vs True Negatives")
print("=" * 70)

comparison_fp = pd.DataFrame({
    "TN (correct)": X_test.loc[tn_mask_full, top5_features].median(),
    "FP (flagged)": X_test.loc[fp_mask_full, top5_features].median(),
})
comparison_fp["Difference"] = comparison_fp["FP (flagged)"] - comparison_fp["TN (correct)"]
comparison_fp["%Diff"] = (comparison_fp["Difference"] / comparison_fp["TN (correct)"].abs().clip(lower=1e-6) * 100).round(1)
print(comparison_fp.to_string(float_format="{:.2f}".format))

print("\nKey insight: False negatives (missed defaults) tend to look more like")
print("non-defaults in top features — they are borderline cases the model finds hard to separate.")

## 6. Business Metrics

In [ ]:
# Define cost model
# Reload original data to get AMT_CREDIT
df_orig = pd.read_parquet(FEATURES_DIR / "train_features.parquet")

if "AMT_CREDIT" in df_orig.columns:
    avg_loan = df_orig["AMT_CREDIT"].mean()
else:
    # Fallback: use a reasonable estimate
    avg_loan = 600_000  # approximate average from EDA

# Cost definitions
cost_fn = avg_loan               # Cost of missing a default (false negative) — full loss
cost_fp = avg_loan * 0.10        # Cost of rejecting a good client (false positive) — lost profit
benefit_tp = avg_loan             # Benefit of catching a default (true positive) — saved loss
cost_tn = 0.0                     # No cost for correctly approving good clients

print("Business Cost Model")
print("=" * 55)
print(f"  Average loan amount (AMT_CREDIT):     {avg_loan:>12,.0f}")
print(f"  Cost of missed default (FN):          {cost_fn:>12,.0f}")
print(f"  Cost of rejecting good client (FP):   {cost_fp:>12,.0f}")
print(f"  Benefit of catching default (TP):     {benefit_tp:>12,.0f}")
print(f"  Cost ratio (FN/FP):                   {cost_fn/cost_fp:>12.1f}x")

In [ ]:
# Calculate expected loss at different thresholds
n_test = len(y_test)
expected_losses = []
expected_profits = []

for t in thresholds:
    preds_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, preds_t)
    tn, fp, fn, tp = cm_t.ravel()

    # Total cost = cost of FN (missed defaults) + cost of FP (rejected good clients)
    total_loss = fn * cost_fn + fp * cost_fp
    # Total benefit = benefit from TP (caught defaults)
    total_benefit = tp * benefit_tp
    # Net = benefit - loss
    net = total_benefit - total_loss

    expected_losses.append(total_loss / n_test)
    expected_profits.append(net / n_test)

expected_losses = np.array(expected_losses)
expected_profits = np.array(expected_profits)

# Plot expected loss and profit curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds, expected_losses, color="crimson", lw=2)
best_loss_idx = np.argmin(expected_losses)
axes[0].axvline(thresholds[best_loss_idx], color="gray", linestyle="--", alpha=0.7,
                label=f"Min loss threshold = {thresholds[best_loss_idx]:.2f}")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Expected Loss per Application")
axes[0].set_title("Expected Loss vs. Threshold")
axes[0].legend()

axes[1].plot(thresholds, expected_profits, color="green", lw=2)
best_profit_idx = np.argmax(expected_profits)
axes[1].axvline(thresholds[best_profit_idx], color="gray", linestyle="--", alpha=0.7,
                label=f"Max profit threshold = {thresholds[best_profit_idx]:.2f}")
axes[1].axhline(0, color="k", linestyle="-", alpha=0.3)
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Expected Net Profit per Application")
axes[1].set_title("Expected Net Profit vs. Threshold")
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_business_metrics.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_business_metrics.png'}")
plt.show()

In [ ]:
# Business summary
thresh_min_loss = thresholds[best_loss_idx]
thresh_max_profit = thresholds[best_profit_idx]

print("Business-Oriented Summary")
print("=" * 65)

for label, t in [("Min Expected Loss", thresh_min_loss), ("Max Expected Profit", thresh_max_profit), ("Max F1", thresh_best_f1)]:
    preds_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, preds_t)
    tn, fp, fn, tp = cm_t.ravel()
    total_loss = (fn * cost_fn + fp * cost_fp) / n_test
    total_benefit = (tp * benefit_tp) / n_test
    net = total_benefit - total_loss

    print(f"\n  {label} (threshold = {t:.2f})")
    print(f"    Defaults caught:   {tp:>6,} / {tp+fn:,} ({tp/(tp+fn)*100:.1f}%)")
    print(f"    Good clients rejected: {fp:>6,} / {tn+fp:,} ({fp/(tn+fp)*100:.1f}%)")
    print(f"    Expected loss/app:    {total_loss:>12,.0f}")
    print(f"    Expected benefit/app: {total_benefit:>12,.0f}")
    print(f"    Net profit/app:       {net:>12,.0f}")

## 7. Calibration

In [ ]:
# Calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy="uniform")
brier = brier_score_loss(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(prob_pred, prob_true, "s-", label=f"LightGBM (Brier = {brier:.4f})", color="steelblue", lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfectly calibrated")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curve")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_calibration_curve.png", dpi=150, bbox_inches="tight")
print(f"Figure saved: {FIGURES_DIR / '08_calibration_curve.png'}")
plt.show()

print(f"\nBrier Score: {brier:.4f}")
print(f"  (lower is better; 0 = perfect, baseline = {y_test.mean()*(1-y_test.mean()):.4f})")

In [ ]:
# Assess calibration quality
# Expected Calibration Error (ECE)
n_bins_ece = 10
bin_edges = np.linspace(0, 1, n_bins_ece + 1)
ece = 0.0
total_samples = len(y_test)

print("Calibration Bin Analysis")
print("=" * 65)
print(f"{'Bin':<15s} {'Count':>8s} {'Mean Pred':>10s} {'Actual Rate':>12s} {'|Gap|':>8s}")
print("-" * 65)

for i in range(n_bins_ece):
    mask = (y_proba >= bin_edges[i]) & (y_proba < bin_edges[i + 1])
    if i == n_bins_ece - 1:  # Include right edge for last bin
        mask = (y_proba >= bin_edges[i]) & (y_proba <= bin_edges[i + 1])
    n_in_bin = mask.sum()
    if n_in_bin > 0:
        mean_pred = y_proba[mask].mean()
        actual_rate = y_test.values[mask].mean()
        gap = abs(actual_rate - mean_pred)
        ece += gap * n_in_bin / total_samples
        print(f"  [{bin_edges[i]:.1f}, {bin_edges[i+1]:.1f})  {n_in_bin:>8,}  {mean_pred:>10.4f}  {actual_rate:>12.4f}  {gap:>8.4f}")

print(f"\nExpected Calibration Error (ECE): {ece:.4f}")

if ece < 0.05:
    print("Calibration is good (ECE < 0.05) — no recalibration needed.")
elif ece < 0.10:
    print("Calibration is acceptable (ECE < 0.10) — consider recalibration for risk-sensitive use.")
else:
    print("Calibration is poor (ECE >= 0.10) — recalibration recommended (e.g., CalibratedClassifierCV).")

## 8. Summary

In [ ]:
print("=" * 70)
print("EVALUATION SUMMARY — NB08")
print("=" * 70)

print("\n--- Model Performance ---")
print(f"  AUC-ROC:  {auc_roc:.4f}")
print(f"  PR-AUC:   {pr_auc:.4f}")
print(f"  Brier:    {brier:.4f}")

print("\n--- Recommended Thresholds ---")
print(f"  Max F1 threshold:         {thresh_best_f1:.2f}")
print(f"  Min business loss:        {thresh_min_loss:.2f}")
print(f"  Max business profit:      {thresh_max_profit:.2f}")
print(f"  Target recall >= 0.7:     {thresh_recall_70:.2f}")

print("\n--- Key Risk Factors (SHAP Top 5) ---")
for i, feat in enumerate(top5_features, 1):
    print(f"  {i}. {feat}")

print("\n--- Model Strengths ---")
print("  - Strong discrimination (AUC-ROC)")
print("  - Interpretable via SHAP — key risk factors align with domain knowledge")
print("  - Business cost model helps select production threshold")

print("\n--- Model Weaknesses ---")
print("  - Class imbalance (8% default) limits precision at high recall")
print("  - False negatives (missed defaults) look similar to non-defaults")
print("  - Threshold choice involves a precision-recall trade-off")

print("\n--- Recommendation ---")
print(f"  Use threshold = {thresh_min_loss:.2f} (min expected business loss) for production.")
print("  Monitor calibration and retrain periodically as data distribution shifts.")

print("\nNext: NB09 — Model Registry & Deployment Prep")
print("=" * 70)

## Summary

**What we did:**
1. Loaded the best LightGBM model and reproduced the train/test split from NB07
2. Computed full classification report, confusion matrix, ROC and PR curves
3. Analyzed precision/recall/F1 across thresholds — found optimal thresholds for different objectives
4. Deep SHAP analysis: beeswarm, bar, dependence, force, and waterfall plots
5. Error analysis: compared feature distributions of TP vs FN and TN vs FP
6. Business cost model: estimated expected loss/profit at different thresholds
7. Calibration assessment: calibration curve, Brier score, ECE

**Key findings:**
- SHAP reveals the most impactful features for default prediction
- False negatives tend to be borderline cases that resemble non-defaults
- The business-optimal threshold differs from the statistical-optimal (max F1) threshold
- Calibration quality determines whether raw probabilities can be used for risk pricing

**Next:** NB09 — Model Registry & Deployment Prep